In [1]:
import requests

In [10]:
ADDRESS = "http://192.168.111.200:9380"
CHAT_ID = 'dc87126e504611f19aceb391bf03f413'
API_KEY = "ragflow-L-Uf9Mrlk7IsJWgS25O5keEqraqudBwUWYfaeuPoSZc"

## 创建会话

In [11]:
url = f"{ADDRESS}/api/v1/chats/{CHAT_ID}/sessions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}",
}

payload = {
    "name": "接口测试"
}

In [16]:
response = requests.post(
    url=url,
    headers=headers,
    json=payload,
    timeout=30,
)
print("HTTP 状态码:", response.status_code)

data = response.json()
print("响应 JSON:", data)


HTTP 状态码: 200
响应 JSON: {'code': 0, 'data': {'chat_id': 'dc87126e504611f19aceb391bf03f413', 'create_date': '2026-05-17T12:37:08', 'create_time': 1778992628824, 'id': '107f022a51aa11f1a5be253d403d7ec5', 'messages': [{'content': '你好！ 我是你的助理，有什么可以帮到你的吗？', 'role': 'assistant'}], 'name': '接口测试', 'update_date': '2026-05-17T12:37:08', 'update_time': 1778992628824, 'user_id': ''}}


In [17]:
data

{'code': 0,
 'data': {'chat_id': 'dc87126e504611f19aceb391bf03f413',
  'create_date': '2026-05-17T12:37:08',
  'create_time': 1778992628824,
  'id': '107f022a51aa11f1a5be253d403d7ec5',
  'messages': [{'content': '你好！ 我是你的助理，有什么可以帮到你的吗？', 'role': 'assistant'}],
  'name': '接口测试',
  'update_date': '2026-05-17T12:37:08',
  'update_time': 1778992628824,
  'user_id': ''}}

# 流式输出测试

In [ ]:
curl --request POST \
     --url http://{address}/api/v1/chats/{chat_id}/completions \
     --header 'Content-Type: application/json' \
     --header 'Authorization: Bearer <YOUR_API_KEY>' \
     --data-binary '
     {
          "question": "Who are you",
          "stream": true,
          "session_id":"9fa7691cb85c11ef9c5f0242ac120005",
          "metadata_condition": {
            "logic": "and",
            "conditions": [
              {
                "name": "author",
                "comparison_operator": "is",
                "value": "bob"
              }
            ]
          }
     }'

In [30]:
ADDRESS = "http://192.168.111.200"
CHAT_ID = 'dc87126e504611f19aceb391bf03f413'
API_KEY = "ragflow-L-Uf9Mrlk7IsJWgS25O5keEqraqudBwUWYfaeuPoSZc"
SESSION_ID = "107f022a51aa11f1a5be253d403d7ec5"

In [25]:
url = f"{ADDRESS}/api/v1/chats/{CHAT_ID}/completions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {API_KEY}",
}

payload = {
    "question": "Who are you",
    "stream": 'true',
    "session_id":SESSION_ID,
    "metadata_condition": {
    "logic": "and",
    "conditions": [
        {
        "name": "author",
        "comparison_operator": "is",
        "value": "bob"
        }
    ]
    }
}

In [28]:
import json
import time
import requests

In [32]:
import json
import requests


# ADDRESS = "http://127.0.0.1:9380"
# API_KEY = "你的_API_KEY"
# CHAT_ID = "你的_CHAT_ID"
# SESSION_ID = "你的_SESSION_ID"


def main():
    url = f"{ADDRESS.rstrip('/')}/api/v1/chats/{CHAT_ID}/completions"

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_KEY}",
    }

    payload = {
        "question": "城市更新主要任务是什么？",
        "stream": True,
        "session_id": SESSION_ID,
    }

    print("请求地址:", url)
    print("请求参数:")
    print(json.dumps(payload, ensure_ascii=False, indent=2))
    print("=" * 80)

    last_answer = ""

    with requests.post(
        url,
        headers=headers,
        json=payload,
        stream=True,
        timeout=(10, 180),
    ) as response:
        print("HTTP 状态码:", response.status_code)
        print("=" * 80)

        if response.status_code != 200:
            print(response.text)
            return

        for line in response.iter_lines(decode_unicode=True):
            if not line:
                continue

            if line.startswith("data:"):
                line = line[len("data:"):].strip()

            if not line:
                continue

            obj = json.loads(line)

            # 流式结束标志
            if obj.get("data") is True:
                print("\n\n[DONE]")
                break

            data = obj.get("data", {})
            answer = data.get("answer", "")

            # RAGFlow 返回的 answer 通常是累计文本，这里只打印新增部分
            if answer:
                if answer.startswith(last_answer):
                    delta = answer[len(last_answer):]
                else:
                    delta = answer

                print(delta, end="", flush=True)
                last_answer = answer

            # 如果有引用，简单打印一下命中文档
            reference = data.get("reference")
            if reference and reference.get("chunks"):
                print("\n\n[命中知识库]")
                for chunk in reference["chunks"]:
                    print("-", chunk.get("document_name"), "similarity:", chunk.get("similarity"))


if __name__ == "__main__":
    main()

请求地址: http://192.168.111.200/api/v1/chats/dc87126e504611f19aceb391bf03f413/completions
请求参数:
{
  "question": "城市更新主要任务是什么？",
  "stream": true,
  "session_id": "107f022a51aa11f1a5be253d403d7ec5"
}
HTTP 状态码: 200
嗯，用户问的是“城市更新主要任务是什么？”，我需要从知识库里找到相关信息来回答。首先，我浏览知识库，发现ID:0和ID:3的内容提到了城市更新的主要任务。ID:0的正文部分明确列出了八项主要任务，包括加强既有建筑改造、推进老旧小区整治、社区建设等。ID:3的切片ID:3也详细说明了这些任务，围绕人民的需求来实施。其他知识库条目主要涉及规划编制和实施机制，但用户的问题更关注任务本身，所以重点放在ID:0和ID:3的内容上。我应该将这些任务逐一列出，并标注相应的引用来源，确保每个任务都有对应的知识库支持。同时，要保持回答的结构化，使用清晰的分点形式，让用户容易理解。


城市更新的主要任务包括：

1. **加强既有建筑改造利用**  
2. **推进城镇老旧小区整治改造**  
3. **开展完整社区建设**  
4. **推进老旧街区、老旧厂区、城中村等更新改造**  
5. **完善城市功能**  
6. **加强城市基础设施建设改造**  
7. **修复城市生态系统**  
8. **保护传承城市历史文化**[ID:0][ID:3]。

这些任务紧密围绕新时期人民需求，立足关键领域精准发力[ID:3]。

[命中知识库]
- 中央明确城市更新“路线图”！宜居、韧性、智慧_20260313.md similarity: 0.5427810629358258
- 节能处-01城市更新-参阅材料-2022-【政策文件】各省城市更新文件-【江西城市更新规划】-附件：江西省城市更新规划编制指南（试行）.md similarity: 0.5375082623780895
- 节能处-01城市更新-参阅材料-2022-【政策文件】各省城市更新文件-【江西城市更新规划】-附件：江西省城市更新规划编制指南（试行）.md similarity: 0.535937863